In [2]:
import csv
import json

with open("/content/qDCCmpyfss1kBJJB.csv", "r", newline="") as file:
    reader = csv.DictReader(file)

    header = reader.fieldnames
    rows = list(reader)

print("Header:", header)
print("Number of rows:", len(rows))


def parse_row(row):
    row["age"] = int(row["age"])
    row["fee_inr"] = int(row["fee_inr"])
    row["prior_no_shows"] = int(row["prior_no_shows"])
    row["distance_km"] = float(row["distance_km"])

    return row


accepted = []
rejected = []

for row in rows:
    try:
        parsed = parse_row(row)
        accepted.append(parsed)
    except ValueError:
        rejected.append(row)

print("Accepted rows:", len(accepted))
print("Rejected rows:", len(rejected))

with open("rejected_rows.json", "w") as file:
    json.dump(rejected, file, indent=2)

Header: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Number of rows: 1020
Accepted rows: 988
Rejected rows: 32


In [3]:
import pandas as pd

df = pd.read_csv("/content/qDCCmpyfss1kBJJB.csv")

print("Shape:", df.shape)

print("\nData Information:")
df.info()

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:")
print(df.duplicated().sum())


# (a)
df = df.drop_duplicates()
print("\nRows after removing duplicates:", len(df))


# (b)
df["booking_channel"] = df["booking_channel"].str.strip().str.title()

print("Rows after cleaning booking channel:", len(df))


# (c)
df["appt_date"] = pd.to_datetime(
    df["appt_date"], errors="coerce"
)

df["booked_on"] = pd.to_datetime(df["booked_on"])

print("Rows after converting dates:", len(df))
#(D)
df = df.dropna(subset=["appt_date", "status"])

print("Rows after dropping missing dates and status:", len(df))

Shape: (1020, 14)

Data Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   object 
 1   patient_id       1020 non-null   object 
 2   clinic_id        1020 non-null   object 
 3   booked_on        1020 non-null   object 
 4   appt_date        1019 non-null   object 
 5   age              1008 non-null   float64
 6   gender           1020 non-null   object 
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   object 
 10  visit_type       1020 non-null   object 
 11  reminder_sent    1020 non-null   object 
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   object 
dtypes: float64(2), int64(2), object(10)
memory usage: 111.7+ KB

Missing Values:
appointment_id      0
patient_id      

In [4]:
import numpy as np

# (a)
df.loc[(df["age"] < 1) | (df["age"] > 100), "age"] = np.nan
print("Missing ages:", df["age"].isnull().sum())

# (b)
age_median = df["age"].median()
distance_median = df["distance_km"].median()
df["age"] = df["age"].fillna(age_median).round().astype(int)
df["distance_km"] = df["distance_km"].fillna(distance_median)
print("Age median:", age_median)
print("Distance median:", distance_median)

# (c)
for col in ["distance_km", "fee_inr"]:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_limit = Q1 - 1.5 * IQR
    upper_limit = Q3 + 1.5 * IQR
    capped = ((df[col] < lower_limit) | (df[col] > upper_limit)).sum()
    df[col] = np.clip(df[col], lower_limit, upper_limit)
    print(f"\n{col}:")
    print("Q1:", Q1)
    print("Q3:", Q3)
    print("IQR:", IQR)
    print("Lower limit:", lower_limit)
    print("Upper limit:", upper_limit)
    print("Number of values capped:", capped)

# (d)
df["distance_km"] = df["distance_km"].round(1)
df["fee_inr"] = df["fee_inr"].round().astype(int)
print("\nLargest fee after capping:", df["fee_inr"].max())

Missing ages: 17
Age median: 48.0
Distance median: 8.25

distance_km:
Q1: 4.625
Q3: 13.0
IQR: 8.375
Lower limit: -7.9375
Upper limit: 25.5625
Number of values capped: 26

fee_inr:
Q1: 500.0
Q3: 787.5
IQR: 287.5
Lower limit: 68.75
Upper limit: 1218.75
Number of values capped: 3

Largest fee after capping: 1219


In [5]:
df["lead_days"] = (df["appt_date"] - df["booked_on"]).dt.days
df["no_show"] = (df["status"] == "No-Show").astype(int)
X = df[["age", "distance_km", "fee_inr", "lead_days"]].to_numpy()
# (a)
print("Mean of each column:")
print(np.mean(X, axis=0))

# (b)
mask = df["lead_days"] > 14
print("\nNumber of appointments with lead_days > 14:", mask.sum())
print("Share of appointments with lead_days > 14:", mask.mean())
print("Percentage:", round(mask.mean() * 100, 1), "%")

# (c)
x = df["distance_km"].to_numpy()
z = (x - np.mean(x)) / np.std(x)
print("\nNumber of rows with |z| > 2:", np.sum(np.abs(z) > 2))

Mean of each column:
[ 48.17373737   9.50090909 635.71414141  11.81919192]

Number of appointments with lead_days > 14: 296
Share of appointments with lead_days > 14: 0.298989898989899
Percentage: 29.9 %

Number of rows with |z| > 2: 53


In [6]:
import requests

# Read the clinic details from the CSV file
clinics = pd.read_csv("/content/jbFzPdd2gB2Ex2TY.csv")

# Get the temperature for each clinic using the Open-Meteo API
def get_temperature(clinic):
    params = {
        "latitude": clinic["latitude"],
        "longitude": clinic["longitude"],
        "hourly": "temperature_2m",
        "forecast_days": 1
    }

    # Try twice in case the API request fails
    for attempt in range(2):
        try:
            response = requests.get(
                "https://api.open-meteo.com/v1/forecast",
                params=params,
                timeout=10
            )

            # Check whether the request was successful
            if response.status_code != 200:
                reason = f"Status code: {response.status_code}"
                continue

            data = response.json()

            # Take the first temperature value from the response
            temperatures = data.get("hourly", {}).get("temperature_2m")

            if temperatures:
                return temperatures[0]

            reason = "Hourly temperature data is missing"

        except requests.exceptions.RequestException as e:
            reason = str(e)

    # If both attempts fail, store None and move to the next clinic
    print(f"Could not get temperature for {clinic['clinic_name']}: {reason}")
    return None


clinics["temp_c"] = clinics.apply(get_temperature, axis=1)
clinics.to_csv("clinics_enriched.csv", index=False)
print("\nClinic details with temperature:")
print(clinics)


Clinic details with temperature:
  clinic_id               clinic_name     city    tier  latitude  longitude  \
0       C01     Kothrud Family Clinic     Pune   Metro   18.5074    73.8077   
1       C02      Hadapsar Care Centre     Pune   Metro   18.5089    73.9260   
2       C03    Sitabuldi Health Point   Nagpur  Tier-2   21.1458    79.0882   
3       C04      Malviya Nagar Clinic   Jaipur  Tier-2   26.8553    75.8150   
4       C05     Vaishali Nagar Clinic   Jaipur  Tier-2   26.9124    75.7873   
5       C06  Edappally Medical Centre    Kochi  Tier-2   10.0261    76.3085   
6       C07        Vijay Nagar Clinic   Indore  Tier-2   22.7533    75.8937   
7       C08    Gomti Nagar Care Point  Lucknow  Tier-2   26.8467    80.9462   

   temp_c  
0    16.2  
1    17.9  
2    20.1  
3    15.5  
4    15.5  
5    25.3  
6    18.1  
7    17.5  


In [7]:
df = df.sort_values("appointment_id")
print("Are appointment IDs unique:", df["appointment_id"].is_unique)
columns = [
    "appointment_id", "patient_id", "clinic_id", "booked_on",
    "appt_date", "age", "gender", "distance_km", "fee_inr",
    "booking_channel", "visit_type", "reminder_sent",
    "prior_no_shows", "status", "lead_days", "no_show"
]

print("\nMissing values in each column:")
print(df[columns].isnull().sum())
print("\nTotal missing values:", df[columns].isnull().sum().sum())
df["booked_on"] = df["booked_on"].dt.strftime("%Y-%m-%d")
df["appt_date"] = df["appt_date"].dt.strftime("%Y-%m-%d")
# Keep only the required 16 columns in the given order
df = df[columns]
# Saving the cleaned appointments data
df.to_csv("appointments_clean.csv", index=False)
print("\nCleaned appointments file saved successfully!")
print("Shape of cleaned data:", df.shape)
# Reading the enriched clinic details
clinics = pd.read_csv("clinics_enriched.csv")
# Create the SQL file
with open("load_data.sql", "w") as file:

    # Insert clinic details first because appointments depend on clinics
    for _, row in clinics.iterrows():

        values = [
            f"'{row['clinic_id']}'",
            f"'{row['clinic_name']}'",
            f"'{row['city']}'",
            f"'{row['tier']}'",
            str(row["latitude"]),
            str(row["longitude"]),
            "NULL" if pd.isna(row["temp_c"]) else str(row["temp_c"])
        ]

        sql = f"INSERT INTO clinics VALUES ({','.join(values)});\n"
        file.write(sql)

    # Insert appointment details after the clinic records
    for _, row in df.iterrows():

        values = []

        for col in columns:
            value = row[col]

            if col in ["age", "distance_km", "fee_inr",
                       "prior_no_shows", "lead_days", "no_show"]:
                values.append(str(value))
            else:
                values.append(f"'{value}'")

        sql = f"INSERT INTO appointments VALUES ({','.join(values)});\n"
        file.write(sql)


print("SQL file created successfully!")
print("Number of clinic INSERT statements:", len(clinics))
print("Number of appointment INSERT statements:", len(df))
print("Total INSERT statements:", len(clinics) + len(df))

Are appointment IDs unique: True

Missing values in each column:
appointment_id     0
patient_id         0
clinic_id          0
booked_on          0
appt_date          0
age                0
gender             0
distance_km        0
fee_inr            0
booking_channel    0
visit_type         0
reminder_sent      0
prior_no_shows     0
status             0
lead_days          0
no_show            0
dtype: int64

Total missing values: 0

Cleaned appointments file saved successfully!
Shape of cleaned data: (990, 16)
SQL file created successfully!
Number of clinic INSERT statements: 8
Number of appointment INSERT statements: 990
Total INSERT statements: 998
